# Testing the Insider Trading Anomaly on Modern Data

**Research Question:** Does a simple, publicly implementable insider cluster-buy strategy generate independent alpha on modern data after controlling for known risk factors?

**Data:** SEC EDGAR Form 4 filings, 2010-2024 (729,122 raw transactions)

**Finding:** After factor adjustment, the strategy's abnormal returns are no longer statistically significant (alpha = 0.08%, t-stat: 0.08). The observed performance is largely explained by market beta and small-cap factor exposure rather than independent alpha.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

trades = pd.read_csv('../data/processed/trades.csv')
trades['signal_date'] = pd.to_datetime(trades['signal_date'])
trades['entry_date'] = pd.to_datetime(trades['entry_date'])
trades['exit_date'] = pd.to_datetime(trades['exit_date'])
trades['year'] = trades['signal_date'].dt.year

print(f'Total trades: {len(trades)}')
print(f'Date range: {trades["signal_date"].min().date()} to {trades["signal_date"].max().date()}')

## 1. Data Pipeline

In [ ]:
funnel = {'Stage': ['Raw Form 4 filings', 'Purchases only', 'Officers/Directors', 'Direct holdings', '$100k+ value', 'Cluster signals (3+, $500k+)', 'Trades with price data'], 'Count': [729122, 213546, 122672, 75802, 34765, 1985, len(trades)]}
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(range(len(funnel['Stage'])), funnel['Count'], color='#2C3E50', alpha=0.85)
ax.set_yticks(range(len(funnel['Stage'])))
ax.set_yticklabels(funnel['Stage'])
ax.invert_yaxis()
ax.set_xlabel('Number of Records')
ax.set_title('Data Pipeline: From Raw Filings to Trading Signals')
for bar, count in zip(bars, funnel['Count']):
    ax.text(bar.get_width() + 5000, bar.get_y() + bar.get_height()/2, f'{count:,}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

## 2. Summary Statistics

In [ ]:
total = len(trades)
winners = (trades['stock_return'] > 0).sum()
print(f'Total Trades:          {total}')
print(f'Winners:               {winners}')
print(f'Losers:                {total - winners}')
print(f'Hit Rate:              {winners/total:.1%}')
print(f'Avg Return:            {trades["stock_return"].mean():.2%}')
print(f'Avg Abnormal (R2000):  {trades["abnormal_return"].mean():.2%}')
print(f'Median Return:         {trades["stock_return"].median():.2%}')
print(f'Sharpe Ratio:          {(trades["stock_return"].mean() / trades["stock_return"].std()) * np.sqrt(252/60):.2f}')
print(f't-stat (abnormal):     {trades["abnormal_return"].mean() / (trades["abnormal_return"].std() / np.sqrt(total)):.2f}')
print(f'\nNote: Sharpe of 0.44 indicates weak risk-adjusted performance.')

## 3. Cumulative Returns

In [ ]:
trades['capped_return'] = trades['stock_return'].clip(-0.5, 1.0)
trades['capped_bench'] = trades['benchmark_return'].clip(-0.5, 1.0)
trades['month'] = trades['entry_date'].dt.to_period('M')
monthly = trades.groupby('month').agg(avg_return=('capped_return', 'mean'), avg_bench=('capped_bench', 'mean')).reset_index()
monthly['month_date'] = monthly['month'].dt.to_timestamp()
cum_s = (1 + monthly['avg_return']).cumprod()
cum_b = (1 + monthly['avg_bench']).cumprod()
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(monthly['month_date'], cum_s, color='#2C3E50', linewidth=2, label='Insider Cluster-Buy Strategy')
ax.plot(monthly['month_date'], cum_b, color='#7F8C8D', linewidth=2, linestyle='--', label='Russell 2000 (same periods)')
ax.fill_between(monthly['month_date'], cum_s, cum_b, where=cum_s >= cum_b, alpha=0.15, color='#27AE60')
ax.fill_between(monthly['month_date'], cum_s, cum_b, where=cum_s < cum_b, alpha=0.15, color='#E74C3C')
ax.axhline(y=1, color='black', linewidth=0.5, alpha=0.3)
ax.set_xlabel('Date')
ax.set_ylabel('Growth of $1')
ax.set_title('Strategy vs Russell 2000 Benchmark')
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Yearly Abnormal Returns

In [ ]:
yearly = trades.groupby('year').agg(trades_count=('stock_return', 'count'), hit_rate=('stock_return', lambda x: (x > 0).mean()), avg_return=('stock_return', 'mean'), avg_abnormal=('abnormal_return', 'mean')).round(4)
print(yearly.to_string())

In [ ]:
yr = trades.groupby('year').agg(avg_abnormal=('abnormal_return', 'mean'))
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#27AE60' if x > 0 else '#E74C3C' for x in yr['avg_abnormal']]
bars = ax.bar(yr.index, yr['avg_abnormal'] * 100, color=colors, width=0.7)
ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_xlabel('Year')
ax.set_ylabel('Average Abnormal Return (%)')
ax.set_title('Yearly Abnormal Returns vs Russell 2000')
ax.set_xticks(yr.index)
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, yr['avg_abnormal'] * 100):
    y_pos = bar.get_height() + 0.3 if bar.get_height() > 0 else bar.get_height() - 0.8
    ax.text(bar.get_x() + bar.get_width()/2, y_pos, f'{val:.1f}%', ha='center', va='bottom' if val > 0 else 'top', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Holding Period Sensitivity

In [ ]:
hp = pd.DataFrame({'Period': ['20 days', '40 days', '60 days', '90 days', '120 days'], 'Hit Rate': ['52.9%', '53.9%', '60.7%', '55.9%', '57.8%'], 'Avg Return': ['0.75%', '2.84%', '6.94%', '8.63%', '10.64%'], 'Avg Abnormal': ['-1.08%', '-0.33%', '2.96%', '1.71%', '1.22%']})
print(hp.to_string(index=False))
print('\n60-day period maximises benchmark-adjusted returns.')
print('Factor-adjusted alpha is near zero regardless of holding period.')

## 6. Fama-French 5-Factor Regression

The critical test: after controlling for market, size, value, profitability, and investment factors, does independent alpha remain?

In [ ]:
factors = pd.read_csv('../data/factors/ff5_daily.csv')
factors['Date'] = pd.to_datetime(factors['Date'])
results = []
for _, trade in trades.iterrows():
    period = factors[(factors['Date'] >= trade['entry_date']) & (factors['Date'] <= trade['exit_date'])]
    if period.empty:
        continue
    r = {'stock_return': trade['stock_return'], 'RF': period['RF'].sum()}
    for f in ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']:
        r[f] = period[f].sum()
    r['excess_return'] = trade['stock_return'] - r['RF']
    results.append(r)
ff_df = pd.DataFrame(results)
y = ff_df['excess_return']
X = sm.add_constant(ff_df[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']])
model = sm.OLS(y, X).fit()
print('Fama-French 5-Factor Regression')
print('=' * 50)
print(f'R-squared: {model.rsquared:.4f}\n')
names = ['Alpha', 'Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']
print(f'{"Factor":<12} {"Coef":>10} {"t-stat":>10} {"p-value":>10}')
print('-' * 44)
for name, coef, tval, pval in zip(names, model.params, model.tvalues, model.pvalues):
    sig = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
    print(f'{name:<12} {coef:>10.4f} {tval:>10.2f} {pval:>10.4f} {sig}')
print(f'\nAlpha = {model.params.iloc[0]:.4f} (t = {model.tvalues.iloc[0]:.2f})')
print(f'Market beta = {model.params["Mkt-RF"]:.2f}, SMB = {model.params["SMB"]:.2f}')
print(f'\nConclusion: Returns are explained by market beta and small-cap exposure.')
print('No statistically significant alpha after factor adjustment.')

## 7. Walk-Forward Validation

In [ ]:
def stats(df, label):
    n = len(df)
    hr = (df['stock_return'] > 0).mean()
    abn = df['abnormal_return'].mean()
    t = abn / (df['abnormal_return'].std() / np.sqrt(n))
    return {'Period': label, 'Trades': n, 'Hit Rate': f'{hr:.1%}', 'Avg Abnormal': f'{abn:.2%}', 't-stat': f'{t:.2f}'}
wf = pd.DataFrame([
    stats(trades, 'Full Sample'),
    stats(trades[trades['year'] <= 2017], 'In-Sample (2010-2017)'),
    stats(trades[trades['year'] >= 2018], 'Out-of-Sample (2018-2024)'),
    stats(trades[(trades['year'] >= 2018) & (trades['year'] != 2020)], 'Out-of-Sample ex-2020'),
    stats(trades[trades['year'] != 2020], 'Full Sample ex-2020'),
])
print(wf.to_string(index=False))
print('\nStatistical significance is driven by crisis-period observations.')

## 8. Return Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(trades['stock_return'].clip(-1, 1) * 100, bins=50, color='#2C3E50', alpha=0.8, edgecolor='white')
axes[0].axvline(x=0, color='red', linestyle='--')
axes[0].axvline(x=trades['stock_return'].mean()*100, color='#27AE60', linewidth=2, label=f'Mean: {trades["stock_return"].mean():.2%}')
axes[0].set_xlabel('Return (%)')
axes[0].set_title('Trade Returns')
axes[0].legend()
axes[1].hist(trades['abnormal_return'].clip(-1, 1) * 100, bins=50, color='#2C3E50', alpha=0.8, edgecolor='white')
axes[1].axvline(x=0, color='red', linestyle='--')
axes[1].axvline(x=trades['abnormal_return'].mean()*100, color='#27AE60', linewidth=2, label=f'Mean: {trades["abnormal_return"].mean():.2%}')
axes[1].set_xlabel('Abnormal Return (%)')
axes[1].set_title('Abnormal Returns')
axes[1].legend()
plt.tight_layout()
plt.show()

## 9. Conclusion

A simple, publicly implementable insider cluster-buy strategy does not generate robust independent alpha in modern US markets after controlling for factor exposures.

**Key findings:**
1. The signal picks winners more often than losers (60.7% hit rate), but returns are explained by market beta (1.28) and small-cap exposure (SMB 1.45) with zero residual alpha
2. Performance is regime-dependent, with the 2020 COVID recovery driving the majority of aggregate abnormal returns
3. The 60-day holding period is critical; at 20 days, abnormal returns are negative
4. This is consistent with recent research indicating insider trading signals have weakened in modern markets, with much of the informational edge occurring prior to public disclosure